# Member - metabolite networks

In [1]:
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from pyvis.network import Network
import json
import pandas as pd

with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)
    
# Color end points for colorFader
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue

d = '#008000'
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)



In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

In [4]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

In [10]:
def cooperation_network(results,metric="frequency",freq_cutoff=0,cutoff=0):#closeness_centrality_met,cutoff=0,use_count=True):

    ## Filter results
    results = results[(results.donor!="environment") & (results.receiver!="environment")]
    results_env = results[results[metric]>freq_cutoff]
    
    # Take mean of every interraction
    results_env = results_env[["donor","compound","receiver",metric]]
    results_grouped = results_env[["donor","compound","receiver",metric]].groupby(["donor","compound","receiver"]).mean()
    results_grouped =results_grouped[results_grouped[metric]>cutoff]
    
    
    # separate contribution of each member in production/consumption (combine + sum)
    index_don = results_grouped.index.droplevel(2)
    results_grouped_don = results_grouped.copy()
    results_grouped_don.index = index_don
    results_grouped_don = results_grouped_don.reset_index().groupby(["donor","compound"]).sum()
    
    index_rec = results_grouped.index.droplevel(0)
    results_grouped_rec =results_grouped.copy()
    results_grouped_rec.index = index_rec
    results_grouped_rec = results_grouped_rec.reset_index().groupby(["compound","receiver"]).sum()
    
    network_data_don = results_grouped_don[metric].copy()
    network_data_rec = results_grouped_rec[metric].copy()
    
    # Create network
    net = Network(directed=True)
    nodes_members = list(set(list(results.receiver.unique())+list(results.donor.unique())))
    color_members = [b2 for node in nodes_members]
    net.add_nodes(nodes_members,color=color_members)

    nodes_mets = list(set(list(network_data_don.index.get_level_values(1).unique())+list(network_data_rec.index.get_level_values(0).unique()) ))
    color_mets = [d for node in nodes_mets]
    net.add_nodes(nodes_mets,color=color_mets)

    maximum_corr = max(network_data_don.max(),network_data_rec.max())
    for index,value in network_data_don.items():

        #net.add_edge(index[0],index[1],width=(value*3),color=colorFader(c1,c2,value/maximum_corr)) 
        net.add_edge(index[0],index[1],width=(value*3),color=c2) 
    
    for index,value in network_data_rec.items():
    
        #net.add_edge(index[0],index[1],width=(value*3),color=colorFader(c1,c2,value/maximum_corr)) 
        net.add_edge(index[0],index[1],width=(value*3),color=c2) 

    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.1)


    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net

In [27]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]*1000
results = results[~results.compound.isin(m9_compounds)]

net = cooperation_network(results,metric="frequency",freq_cutoff=0.0,cutoff=0.5)
net.save_graph("figures/network_member_met_frequency_NONE.html")
net.show("network_file.html")


network_file.html


In [29]:
net = cooperation_network(results,metric="mass_rate*frequency",freq_cutoff=0.0,cutoff=0.5)
net.save_graph("figures/network_member_met_massfreq_NONE.html")
net.show("network_file.html")

network_file.html


In [18]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/BH_minus_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]*1000
results = results[~results.compound.isin(m9_compounds)]

In [20]:
net = cooperation_network(results,metric="mass_rate*frequency",freq_cutoff=0.0,cutoff=0.5)
net.show("network_file.html")

network_file.html


In [30]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/ER_minus_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]

In [31]:
net = cooperation_network(results,freq_cutoff=0.0,cutoff=0.3)
net.show("network_file.html")

network_file.html


In [32]:
results = pd.read_csv("../output/Venturelli2018/results_steadiercom/EL_minus_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]

In [33]:
net = cooperation_network(results,freq_cutoff=0.0,cutoff=0.4)
net.show("network_file.html")

network_file.html
